# AI Visual Quality Inspection: Dataset Exploration and Preprocessing

## Kaggle Dataset Instructions
We are using the **Casting Product Image Data for Quality Inspection** dataset from Kaggle.

**How to download and organize:**
1. Go to [Kaggle Casting Product Image Dataset](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product)
2. Download the dataset zip file.
3. Extract the contents and place them into the `data/raw/` directory of this project.
4. The structure should look like this:
```
data/raw/
├── def_front/
│   ├── cast_def_0_...jpeg
│   └── ...
└── ok_front/
    ├── cast_ok_0_...jpeg
    └── ...
```
*(Note: Depending on the extraction, you might need to merge train/test folders from Kaggle into a single structure under raw to perform our own custom splits, or just point `DATA_DIR` to the Kaggle train folder for exploration).* 

In [ ]:
import os
import sys
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add the project root to sys.path to import our custom modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from app.ml.data_loader import get_image_paths_and_labels
from app.ml.preprocessing import load_and_preprocess_image, split_dataset

## 1. Load Dataset Metadata

In [ ]:
# Point to the raw data directory. 
# Ensure the dataset is extracted here before running.
DATA_DIR = os.path.join(project_root, 'data', 'raw')

# Load image paths and labels using our data loader
image_paths, labels, idx_to_class = get_image_paths_and_labels(DATA_DIR)

print(f"Total images found: {len(image_paths)}")
print(f"Classes mapping: {idx_to_class}")

## 2. Dataset Statistics and Class Distribution

In [ ]:
if labels:
    # Calculate distribution
    class_counts = Counter(labels)
    
    print("Class Distribution:")
    for idx, count in class_counts.items():
        print(f"- {idx_to_class[idx]}: {count} images ({count/len(labels)*100:.2f}%)")
        
    # Plot the distribution
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 5))
    ax = sns.barplot(x=[idx_to_class[i] for i in class_counts.keys()], y=list(class_counts.values()))
    plt.title("Class Distribution in Casting Dataset")
    plt.ylabel("Number of Images")
    plt.xlabel("Class")
    plt.show()
else:
    print("No data found. Please ensure the dataset is placed in data/raw/")

## 3. Display Sample Images

In [ ]:
def show_samples(image_paths, labels, idx_to_class, num_samples_per_class=3):
    """Displays random sample images from each class."""
    if not image_paths: return
    
    classes = list(idx_to_class.keys())
    fig, axes = plt.subplots(len(classes), num_samples_per_class, figsize=(12, 4 * len(classes)))
    
    import random
    
    for row, cls_idx in enumerate(classes):
        # Get all image paths for this class
        cls_paths = [p for p, l in zip(image_paths, labels) if l == cls_idx]
        
        # Randomly select samples
        samples = random.sample(cls_paths, min(num_samples_per_class, len(cls_paths)))
        
        for col, img_path in enumerate(samples):
            ax = axes[row, col] if len(classes) > 1 else axes[col]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f"{idx_to_class[cls_idx]} ({os.path.basename(img_path)})")
            ax.axis('off')
            
    plt.tight_layout()
    plt.show()

show_samples(image_paths, labels, idx_to_class)

## 4. Test Preprocessing Pipeline

We will load an image through our preprocessing pipeline to verify resizing and normalization (0 to 1 scaling).

In [ ]:
if image_paths:
    sample_path = image_paths[0]
    print(f"Testing preprocessing on: {sample_path}")
    
    target_size = (300, 300)
    processed_img = load_and_preprocess_image(sample_path, target_size=target_size)
    
    print(f"Processed image shape: {processed_img.shape} (H, W, Channels)")
    print(f"Processed image data type: {processed_img.dtype}")
    print(f"Min pixel value: {processed_img.min()}")
    print(f"Max pixel value: {processed_img.max()}")
    
    # Visualize processed image (needs squeeze since it has channel dimension)
    plt.imshow(processed_img.squeeze(), cmap='gray')
    plt.title("Preprocessed Image")
    plt.axis('off')
    plt.show()

## 5. Train / Validation / Test Split

We split the dataset using an 70% Train, 15% Validation, and 15% Test configuration.

In [ ]:
if image_paths:
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        image_paths, labels, test_size=0.15, val_size=0.15
    )
    
    print("Dataset Split Summary:")
    print(f"Training set: {len(X_train)} samples ({len(X_train)/len(image_paths)*100:.1f}%)")
    print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(image_paths)*100:.1f}%)")
    print(f"Test set: {len(X_test)} samples ({len(X_test)/len(image_paths)*100:.1f}%)")
    
    # Save metadata for next milestones if necessary (e.g. JSON or CSV containing paths and labels)
    # For example: saving split indices or paths to a CSV in data/processed/
    print("\nReady for model building in Milestone 2!")